# 5장 1강: 배깅과 랜덤포레스트

## 실습 목표
이번 실습에서는 **Ames Housing** 데이터를 이용하여 배깅과 랜덤포레스트의 핵심 원리를 확인합니다.

- 하나의 의사결정나무와 여러 나무를 모은 배깅의 차이를 확인합니다.
- 배깅에서 여러 회귀 모델의 예측을 평균내는 이유를 이해합니다.
- 랜덤포레스트의 부트스트랩과 피처 무작위 선택 개념을 확인합니다.
- OOB Score와 Feature Importance를 확인합니다.
- 여러 나무를 평균낼 때 분산이 줄어드는 이유를 설명합니다.

> 교안에 나온 범위만 사용합니다.

## 실습 환경 / 데이터
- Python, pandas, numpy, matplotlib, scikit-learn
- 데이터: `data/Ames_Housing.csv`
- 예측 대상: `SalePrice`

사용 특성:
`OverallQual`, `GrLivArea`, `GarageCars`, `GarageArea`, `TotalBsmtSF`, `1stFlrSF`,
`2ndFlrSF`, `FullBath`, `TotRmsAbvGrd`, `YearBuilt`, `YearRemodAdd`, `Fireplaces`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import r2_score

house = pd.read_csv("data/Ames_Housing.csv")

features = [
    "OverallQual","GrLivArea","GarageCars","GarageArea",
    "TotalBsmtSF","1stFlrSF","2ndFlrSF","FullBath",
    "TotRmsAbvGrd","YearBuilt","YearRemodAdd","Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
display(X.isna().sum().to_frame("missing"))

# 필수 1. 하나의 나무와 Bagging 비교

## 문제 1. Decision Tree 하나를 학습하세요.
1. `DecisionTreeRegressor(random_state=42)` 사용
2. Train 학습
3. Train/Test 예측
4. Train R², Test R² 계산
5. 두 점수 차이 해석

In [ ]:
# TODO
# Decision Tree 하나 학습 및 Train/Test R²
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_train_pred = tree.predict(X_train)
y_test_pred = tree.predict(X_test)

print("Train R²:", r2_score(y_train, y_train_pred))
print("Test R²:", r2_score(y_test, y_test_pred))


### Q1. Train R²는 매우 높은데 Test R²가 더 낮다면 무엇을 의미하나요?

**답변:** 학습 데이터에 과적합되어 일반화 성능이 떨어졌다는 신호입니다.


## 문제 2. Bagging을 적용하세요.
1. 기본 모델은 `DecisionTreeRegressor(random_state=42)`
2. `BaggingRegressor`
3. `n_estimators=50`
4. `random_state=42`
5. Test R² 계산
6. `max_features="sqrt`
7. Decision Tree 하나와 비교

In [ ]:
# TODO
base_tree = DecisionTreeRegressor(random_state=42)
try:
    bag = BaggingRegressor(
        estimator=base_tree,
        n_estimators=50,
        max_features="sqrt",
        random_state=42,
    )
except TypeError:
    bag = BaggingRegressor(
        base_estimator=base_tree,
        n_estimators=50,
        max_features="sqrt",
        random_state=42,
    )
bag.fit(X_train, y_train)
print("Bagging Test R²:", r2_score(y_test, bag.predict(X_test)))
print("Single Tree Test R²:", r2_score(y_test, tree.predict(X_test)))


### Q2. 왜 각 나무를 서로 다른 부트스트랩 샘플로 학습하나요?

**답변:** 모델 간 다양성을 높여 예측을 평균할 때 분산을 줄이기 위함입니다.


# 필수 2. Random Forest의 OOB Score와 Feature Importance

## 문제 1. OOB Score를 확인하세요.
1. `RandomForestRegressor`
2. `n_estimators=100`
3. `bootstrap=True`
4. `oob_score=True`
5. `random_state=42`
6. Test R²와 `oob_score_` 출력

In [ ]:
# TODO
rf = RandomForestRegressor(
    n_estimators=100,
    bootstrap=True,
    oob_score=True,
    random_state=42,
)
rf.fit(X_train, y_train)
print("OOB Score:", rf.oob_score_)
print("Test R²:", r2_score(y_test, rf.predict(X_test)))


### Q3. OOB 샘플은 무엇인가요?

**답변:** 부트스트랩에서 뽑히지 않아 해당 나무 학습에 쓰이지 않은 샘플입니다.

### Q4. OOB Score가 교차검증을 완전히 대체하나요?

**답변:** 아니요. 빠른 내부 추정에는 유용하지만 공식적인 CV/홀드아웃을 완전히 대체하지는 않습니다.


## 문제 2. Feature Importance를 확인하세요.
1. `feature_importances_` 사용
2. 특성명과 중요도 DataFrame 생성
3. 내림차순 정렬
4. 막대그래프
5. 가장 높은 중요도 특성 확인

In [ ]:
# TODO
imp_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)
display(imp_df)

plt.figure(figsize=(8, 5))
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

print("가장 중요한 특성:", imp_df.iloc[0]["feature"])


### Q5. Feature Importance가 높다는 것이 그 변수가 주택가격의 원인이라는 뜻인가요?

**답변:** 아닙니다. 모델이 분할에 많이 사용했다는 상대적 중요도일 뿐 인과를 의미하지 않습니다.


# 과제 1. 나무 수와 예측 안정성 확인

`tree_counts = [1, 10, 50, 100]`을 사용하세요.

각 나무 수에서:
1. RandomForestRegressor 생성
2. `bootstrap=True`, `random_state=42`
3. Train 학습
4. Test R² 계산
5. DataFrame 정리
6. 나무 수에 따른 성능 변화 확인
7. 여러 예측의 평균과 분산 감소 관계 설명

In [ ]:
# TODO
tree_counts = [1, 10, 50, 100]
rows = []
for n in tree_counts:
    model = RandomForestRegressor(
        n_estimators=n,
        bootstrap=True,
        random_state=42,
    )
    model.fit(X_train, y_train)
    rows.append({
        "n_estimators": n,
        "test_r2": r2_score(y_test, model.predict(X_test)),
    })
stability_df = pd.DataFrame(rows)
display(stability_df)

plt.figure(figsize=(7, 4))
plt.plot(stability_df["n_estimators"], stability_df["test_r2"], marker="o")
plt.xlabel("n_estimators")
plt.ylabel("Test R²")
plt.title("Tree count vs Test R²")
plt.tight_layout()
plt.show()


### Q6. 나무 수를 늘리면 왜 예측이 더 안정적으로 변하나요?

**답변:** 여러 나무 예측을 평균하면서 개별 나무의 분산이 상쇄되기 때문입니다.

### Q7. 나무 수를 계속 늘리면 분산이 무한히 0으로 줄어드나요?

**답변:** 아니요. 어느 수준 이후 이득이 체감되고, 트리 간 상관 등으로 분산이 0까지 줄지는 않습니다.


# 실습 마무리
1. 부트스트랩은 복원 추출로 같은 크기 표본을 반복 만드는 방식입니다.
2. Bagging 회귀는 여러 모델 예측을 평균합니다.
3. Random Forest는 분할 후보 특성도 무작위로 고릅니다.
4. OOB 샘플은 평균적으로 약 37%입니다.
5. Feature Importance가 높다고 인과를 의미하지는 않습니다.
